<h2 style="text-align: center">CUSTOMER SEGMENATION</h2>

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import pandera.pandas as pa
import matplotlib.pyplot as plt
import joblib
from scipy import stats
from scipy.stats import kruskal
from sklearn.metrics import davies_bouldin_score
from sklearn.metrics import adjusted_rand_score
from sklearn.preprocessing import PowerTransformer
from sklearn.metrics import silhouette_samples
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from matplotlib.patches import Patch, Ellipse
from sklearn.decomposition import PCA
from sklearn.mixture import GaussianMixture
from sklearn.utils import resample



pd.set_option('display.float_format', '{:.2f}'.format)

## DATA LOADING

In [ ]:
DATA_PATH = "../data/online_retail_II.xlsx"

df = pd.read_excel(DATA_PATH, sheet_name=None, dtype={"Customer ID": str})
df = pd.concat(df.values())

## DATA INSPECTION

**Data preview**

In [ ]:
df.head(10)

- Mix of categorical and numerical columns

**Missing values**

In [ ]:
df.info()

- Only `Description` and `Customer ID` have missing values.

**Pre-schema validation**

In [ ]:
# Schema validation before data cleaning (so nulls are allowed) - just for contract check
# to make surethe Excel loaded with the expected columns and types before


schema_raw = pa.DataFrameSchema({
    "Customer ID": pa.Column(str, nullable=True),
    "Invoice": pa.Column(object, nullable=True),
    "Quantity": pa.Column(int, nullable=True),
    "Price": pa.Column(float, nullable=True),
    "InvoiceDate": pa.Column("datetime64[us]", nullable=True),
    "Country": pa.Column(str, nullable=True),
    "Description": pa.Column(object, nullable=True),
    "StockCode": pa.Column(object, nullable=True)
})
schema_raw.validate(df)

**Value ranges and distributions**

In [ ]:
df.describe()

- Negative values in `Quantity` and `Price` columns

In [ ]:
df.describe(include=['object', 'str'])

In [ ]:
len(df[df["Price"] == 0])

- 6,202 rows with `Price == 0` — free items or data entry errors, because of the big relativie insignificance compared to entire dataset, it will be filtered out.

 **Country distribution**

In [ ]:
print(df["Country"].value_counts().head(10))

- UK dominates with 981,330 transactions (~92%) — model will primarily reflect UK customer behavior.

**Inspecting missing Customer ID rows**

In [ ]:
df[df["Customer ID"].isna()].head(10)

**Inspecting negative values**

In [ ]:
df[df["Quantity"] < 0].head(10)

In [ ]:
df[df["Price"] < 0].head(10)

**Invoice type inspection**

In [ ]:
df["Invoice"] = df["Invoice"].astype("str")

df[df["Invoice"].str.match("^\\d{6}$") == False]

In [ ]:
# Unique Invoice prefixes values

df["Invoice"].str.replace("[0-9]", "", regex=True).unique()

In [ ]:
df[df["Invoice"].str.startswith("A", na = False)]

**StockCode type inspection**

In [ ]:
# This regex is for StockCode values that are not 5 digits or 5 digits followed by letters, which are the valid formats for StockCode. 
# So it reveals all the anomalous/non-standard stock codes

df["StockCode"] = df["StockCode"].astype("str")

df[(df["StockCode"].str.match("^\\d{5}$") == False) & (df["StockCode"].str.match("^\\d{5}[a-zA-Z]+$") == False)]["StockCode"].unique()

In [ ]:
df[df["StockCode"].str.contains("^DOT")]

## DATA CLEANING
Cleaning starts here — all operations on `cleaned_df` preserve the original `df`.

In [ ]:
cleaned_df = df.copy()

**Filtering invalid invoices**

In [ ]:
mask = (
    cleaned_df["Invoice"].str.match("^\\d{6}$") == True
    )

cleaned_df = cleaned_df[mask]

cleaned_df

**Filtering invalid stock codes**

In [ ]:
mask = (
    (cleaned_df["StockCode"].str.match("^\\d{5}$") == True)
    | (cleaned_df["StockCode"].str.match("^\\d{5}[a-zA-Z]+$") == True)
)

cleaned_df = cleaned_df[mask]

cleaned_df

- Description, stockcode are never used after data cleaning in this project, so there is no point in filtering them, likewise country is also never used because it would be heavily distorted by one country (UK).

**Filtering invalid values**

In [ ]:
cleaned_df = cleaned_df[cleaned_df["Price"] > 0]

cleaned_df = cleaned_df[cleaned_df["Quantity"] > 0]

**Dropping missing values**

In [ ]:
cleaned_df.dropna(subset=["Customer ID"], inplace=True)

**Dropping duplicates**

In [ ]:
cleaned_df.drop_duplicates(inplace=True) 

**Post-cleaning summary**

In [ ]:
cleaned_df.describe()

**Post-schema validation**

In [ ]:
# Schema validation after cleaning (so nulls are not allowed)

schema_cleaned = pa.DataFrameSchema({
    "Customer ID": pa.Column(str, nullable=False),
    "Invoice": pa.Column(str, nullable=False),
    "Quantity": pa.Column(int, pa.Check.greater_than(0), nullable=False),
    "Price": pa.Column(float, pa.Check.greater_than(0), nullable=False),
    "Description": pa.Column(object, nullable=False),
    "StockCode": pa.Column(str, nullable=False),
    "InvoiceDate": pa.Column("datetime64[us]", nullable=False),
    "Country": pa.Column(str, nullable=False),
})
schema_cleaned.validate(cleaned_df)

**Cleaning summary**

In [ ]:
len(cleaned_df)/len(df)

- dropped about 27 % of the records.

## FEATURE ENGINEERING

**FEATURES**: Recency, Frequency, MonetaryValue, AOV, Tenure

In [ ]:
# Computing total revenue per transaction line before aggregating to customer level.

cleaned_df["SalesLineTotal"] = cleaned_df["Quantity"] * cleaned_df["Price"]

cleaned_df

In [ ]:
#  Aggregating transactions to customer level — one row per customer with total spend, order count, and first/last purchase date.

aggregated_df = cleaned_df.groupby(by="Customer ID", as_index=False) \
    .agg(
        MonetaryValue=("SalesLineTotal", "sum"),
        Frequency=("Invoice", "nunique"),
        LastInvoiceDate=("InvoiceDate", "max"),
        FirstInvoiceDate=("InvoiceDate", "min")
    )

aggregated_df.head(5)

In [ ]:
# Deriving Recency, Tenure and AOV from aggregated data
max_invoice_date = aggregated_df["LastInvoiceDate"].max()

aggregated_df["Recency"] = (max_invoice_date - aggregated_df["LastInvoiceDate"]).dt.days
aggregated_df["Tenure"] = (aggregated_df["LastInvoiceDate"] - aggregated_df["FirstInvoiceDate"]).dt.days


# dropping intermediate date columns.
aggregated_df["AOV"] = aggregated_df["MonetaryValue"] / aggregated_df["Frequency"]
aggregated_df.drop(columns=["LastInvoiceDate", "FirstInvoiceDate"], inplace=True)

aggregated_df.head(5)

**Feature definitions**

- `MonetaryValue` = total revenue per customer across all orders.
- `Frequency` = number of unique invoices (orders) per customer.
- `Recency` = days since last purchase (from dataset's last date as reference).
- `Tenure` = days between first and last purchase — proxy for customer lifetime.
- `AOV` = `MonetaryValue` / `Frequency` — average spend per order.

## FEATURE DISTRIBUTION ANALYSIS

**Skewness and kurtosis**

**Note**
- At that scal any normality test will almost always reject normality even for trivially small deviations, simply due to statistical power and sampling could give little randomness concerns, so descriptive statistics are used.

In [ ]:
# skewness and kurtosis descriptive stats
print("SKEWNESS")
print(aggregated_df[['MonetaryValue', 'Frequency', 'Recency', 'Tenure', 'AOV']].skew())

print()

print("KURTOSIS")
print(aggregated_df[['MonetaryValue', 'Frequency', 'Recency', 'Tenure', 'AOV']].kurtosis())

- `MonetaryValue` (skew 25.33, kurtosis 849.49), `Frequency` (12.04, 240.20), and `AOV` (53.38, 3446.38) — severely right-skewed with extremely fat tails. Transformation is critical.

- `Recency` (skew 0.89, kurtosis -0.46) — mild skew, slightly platykurtic (flat tails). Minor issue.

- `Tenure` (skew 0.39, kurtosis -1.35) — near normal skew, platykurtic — least problematic.

In [ ]:
# skewness and kurtosis visualization using histograms and Q-Q plots

fig, axes = plt.subplots(2, 5, figsize=(20, 8))
cols = ['MonetaryValue', 'Frequency', 'Recency', 'Tenure', 'AOV']

for i, col in enumerate(cols):
    axes[0, i].hist(aggregated_df[col], bins=30, edgecolor='white')
    axes[0, i].set_title(col, fontsize=13)
    axes[0, i].spines['top'].set_visible(False)
    axes[0, i].spines['right'].set_visible(False)

    stats.probplot(aggregated_df[col], plot=axes[1, i])
    axes[1, i].set_title(f'{col} Q-Q Plot', fontsize=13)
    axes[1, i].spines['top'].set_visible(False)
    axes[1, i].spines['right'].set_visible(False)

axes[0, 0].set_ylabel('Skewness (Histograms)', fontsize=13)
axes[1, 0].set_ylabel('Kurtosis (Q-Q Plots)', fontsize=13)

plt.suptitle('Distributions and Q-Q Plots Before Transformation', fontsize=15)
plt.tight_layout()
plt.show()

- Histograms confirm `MonetaryValue`, `Frequency`, and `AOV` are severely right-skewed with most values concentrated near zero; `Recency` and `Tenure` have mid-skew.

- Q-Q plots confirm `MonetaryValue`, `Frequency`, and `AOV` have extremely heavy tails (sharp upward deviation); `Recency` and `Tenure` show an S-curve — light tails, consistent with their negative kurtosis.

**Outliers**

In [ ]:
# outlier statistics using IQR method (IQR because of skewness)

features = {
    "Recency": aggregated_df["Recency"],
    "MonetaryValue": aggregated_df["MonetaryValue"],
    "Frequency": aggregated_df["Frequency"],
    "Tenure": aggregated_df["Tenure"],
    "AOV": aggregated_df["AOV"],
}

for name, series in features.items():
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1

    mild_threshold = Q3 + 1.5 * IQR
    extreme_threshold = Q3 + 3.0 * IQR

    # All outliers
    outliers = aggregated_df[series > mild_threshold]

    # Mild only
    mild_outliers = aggregated_df[(series > mild_threshold) & (series <= extreme_threshold)]

    # Extreme
    extreme_outliers = aggregated_df[series > extreme_threshold]

    print(f"\n--- {name} ---")
    print(f"Total outliers (1.5×IQR): {len(outliers)}")
    print(f"Mild outliers: {len(mild_outliers)}")
    print(f"Extreme outliers: {len(extreme_outliers)}")

    if len(outliers) > 0:
        print(outliers[name].describe())

- `Recency` and `Tenure` — no outliers, high variance is natural spread in customer behavior.

- `MonetaryValue` — 619 outliers (366 extreme), max £580k — wholesale/B2B buyers.

- `Frequency` — 424 outliers, max 373 orders — confirms high-volume buyers.

- `AOV` — 384 outliers, max £84k per order — extreme per-order spenders.


**Conclusion**: `MonetaryValue`, `Frequency`, `AOV` need outlier handling

In [ ]:
# Outlier visualization using boxplots

fig, axes = plt.subplots(1, 5, figsize=(20, 8))
cols = ['MonetaryValue', 'Frequency', 'Recency', 'Tenure', 'AOV']

for ax, col in zip(axes, cols):
    sns.boxplot(data=aggregated_df[col], ax=ax)
    ax.set_title(col, fontsize=13)
    ax.set_xlabel('')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.suptitle('Feature Outliers Before Transformation', fontsize=15)
plt.tight_layout()
plt.show()


- Whiskers visually confirm descriptive statistics: `MonetaryValue`, `Frequency`, `AOV` have significant outliers, `Recency`, `Tenure` have no outliers

**Correlation Matrix**

In [ ]:
plt.figure(figsize=(10, 8))
corr = aggregated_df[['Recency', 'Frequency', 'MonetaryValue', 'Tenure', 'AOV']].corr()

sns.heatmap(
    corr,
    annot=True,
    fmt=".2f",
    cmap='coolwarm',
    vmin=-1,
    vmax=1,
    linewidths=0.5,
    square=True,
    cbar_kws={"shrink": .8}
)

plt.title("Correlation Matrix of RFM + Tenure + AOV Features - Before transformations", fontsize=14, pad=20)
plt.show()

- `Frequency` ↔ `MonetaryValue`: 0.62 — mild multicollinearity, customers who buy more also spend more.
- `Recency` ↔ `Tenure`: -0.56 — moderate negative, longer-tenure customers tend to be more recent.
- All other pairs below 0.5 — no strong multicollinearity at this stage.

**FINAL CONCLUSIONS OF FEATURE DISTRIBUTION ANALYSIS:** 
- Most features are heavily right-skewed - transformation is needed.
- Most features have significant outliers - outlier removal is needed.
- Data is on different scales - normalization is needed.
- Mild multicolinearity for frequency-monetary, but it's below 0.7 threshold.

## OUTLIER HANDLING

In [ ]:
M_Q1 = aggregated_df["MonetaryValue"].quantile(0.25)
M_Q3 = aggregated_df["MonetaryValue"].quantile(0.75)
M_IQR = M_Q3 - M_Q1


monetary_outliers_df = aggregated_df[(aggregated_df["MonetaryValue"] > (M_Q3 + 1.5 * M_IQR)) 
                                     
                                    | (aggregated_df["MonetaryValue"] < (M_Q1 - 1.5 * M_IQR))].copy()

monetary_outliers_df.describe()

In [ ]:
F_Q1 = aggregated_df['Frequency'].quantile(0.25)
F_Q3 = aggregated_df['Frequency'].quantile(0.75)
F_IQR = F_Q3 - F_Q1


frequency_outliers_df = aggregated_df[(aggregated_df['Frequency'] > (F_Q3 + 1.5 * F_IQR)) 
                                      | (aggregated_df['Frequency'] < (F_Q1 - 1.5 * F_IQR))].copy()

frequency_outliers_df.describe()

In [ ]:
AOV_Q1 = aggregated_df['AOV'].quantile(0.25)
AOV_Q3 = aggregated_df['AOV'].quantile(0.75)
AOV_IQR = AOV_Q3 - AOV_Q1

aov_outliers_df = aggregated_df[(aggregated_df['AOV'] > (AOV_Q3 + 1.5 * AOV_IQR)) 
                                | (aggregated_df['AOV'] < (AOV_Q1 - 1.5 * AOV_IQR))].copy()

aov_outliers_df.describe()

In [ ]:
non_outliers_df = aggregated_df[
    (~aggregated_df.index.isin(monetary_outliers_df.index)) & 
    (~aggregated_df.index.isin(frequency_outliers_df.index)) &
    (~aggregated_df.index.isin(aov_outliers_df.index))].copy()
non_outliers_df.describe()

In [ ]:
plt.figure(figsize=(20, 10))

for i, col in enumerate(["MonetaryValue", "Frequency", "Recency", "AOV", "Tenure"], 1):
    plt.subplot(1, 5, i)
    sns.boxplot(data=non_outliers_df[col], color='skyblue')
    plt.title(f'{col} After Outlier Removal')

plt.tight_layout()
plt.show()

**CONCLUSION:**  The number of outliers has decreased, the remaining outliers - power transformation should handle by compressing extreme values toward the center so it will not distort the algorithm.

## POWER TRANSFORMATION:

In [ ]:
cols = ["MonetaryValue", "Frequency", "Recency", "Tenure", "AOV"]
print((non_outliers_df[cols] == 0).sum())

- Box-Cox transformation is not suitable, because there are 0 values , therefore Yeo-Johnson is most appropriate which handles it naturally.

In [ ]:
cols = ["MonetaryValue", "Frequency", "Recency", "Tenure", "AOV"]
pt = PowerTransformer(method='yeo-johnson')
transformed_df = non_outliers_df[cols].copy()
transformed_df[cols] = pt.fit_transform(non_outliers_df[cols])

## POST-POWER TRANSFORMATION INSPECTION

**Skewness**

In [ ]:
cols = ['MonetaryValue', 'Frequency', 'Recency', 'AOV', 'Tenure']
print(transformed_df[cols].skew())

- All features show |skew| < 0.5 (Tenure highest at -0.33) — below the threshold for meaningful skew, so considered negligible. Yeo-Johnson normalized the distributions effectively; formal skew tests were skipped as unreliable at this sample size.

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(25, 4))
cols = ['MonetaryValue', 'Frequency', 'Recency', 'AOV', 'Tenure']

for i, col in enumerate(cols):
    axes[i].hist(transformed_df[col], bins=30, edgecolor='white')
    axes[i].set_title(col, fontsize=13)
    axes[i].spines['top'].set_visible(False)
    axes[i].spines['right'].set_visible(False)

axes[0].set_ylabel('Frequency', fontsize=13)
plt.suptitle('Histograms After Transformation (Skewness)', fontsize=15)
plt.tight_layout()
plt.show()

- Clearly visible right skew for `Tenure`, `Recency` and `Frequency`, the rest is moderately symmetrical

**Kurtosis**

In [ ]:
cols = ['MonetaryValue', 'Frequency', 'Recency', 'AOV', 'Tenure']

print(transformed_df[cols].kurtosis())

- All features show negative excess kurtosis (platykurtic — flatter than normal, no heavy tails). `AOV` (-0.11) and `MonetaryValue` (-0.53) are closest to normal; `Frequency`, `Recency`, and `Tenure` (-1.2 to -1.56) are more pronounced, driven by their stepped nature (e.g. integer order counts, Tenure's spike at 0 days) rather than by extreme outliers.

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(25, 4))

for i, col in enumerate(cols):
    stats.probplot(transformed_df[col], plot=axes[i])
    axes[i].set_title(f'{col} Q-Q Plot', fontsize=13)
    axes[i].spines['top'].set_visible(False)
    axes[i].spines['right'].set_visible(False)

plt.suptitle('Q-Q Plots After Transformation (Kurtosis)', fontsize=15)
plt.tight_layout()
plt.show()

- Histograms confirm transformation improved distributions — `MonetaryValue` and `AOV` are approximately bell-shaped; `Frequency` and `Tenure` retain non-normal patterns due to their discrete/stepped nature.

- Q-Q plots show `MonetaryValue` and `AOV` closely follow the diagonal — near normal. `Frequency` and `Tenure` deviate significantly — consistent with known limitations: Frequency's gradual decay and Tenure's massive 0-day spike.

- Skewness statistics alone were misleading — `Frequency` (0.12) and `Tenure` (-0.33) appeared near-normal, but histograms reveal discrete stepped patterns and a massive 0-day spike in `Tenure` that statistics failed to capture.

In [ ]:
non_outliers_df["Frequency"].value_counts().head(10)

- `Frequency`  is a gradual decay - natural customer behavior, where most buy infrequently, less impactful for clusters.

In [ ]:
non_outliers_df["Tenure"].value_counts().head(10)

- `Tenure` has a massive spike at 0 days (~1562 customers) - problematic.

**Outliers**

In [ ]:
# Outlier detection after transformation using IQR
# IQR is distribution-agnostic — appropriate here as Frequency, Tenure and Recency
# remain non-normal after Yeo-Johnson (discrete patterns, uniform spread visible in Q-Q plots)

for col in ['MonetaryValue', 'Frequency', 'Recency', 'AOV', 'Tenure']:
    Q1 = transformed_df[col].quantile(0.25)
    Q3 = transformed_df[col].quantile(0.75)
    IQR = Q3 - Q1
    outliers = transformed_df[(transformed_df[col] < Q1 - 1.5 * IQR) | 
                               (transformed_df[col] > Q3 + 1.5 * IQR)]
    print(f"{col}: {len(outliers)} outliers")

- After Yeo-Johnson transformation only 11 outliers remain in `MonetaryValue` and 20 in `AOV` — transformation effectively compressed extreme values. No action needed, they won't meaningfully distort cluster centroids.

In [ ]:
for col in ['MonetaryValue', 'AOV']:
    Q1, Q3 = transformed_df[col].quantile([0.25, 0.75])
    mask = transformed_df[col] < Q1 - 1.5 * (Q3 - Q1)
    print(f"\n{col} lower outliers (raw values):")
    print(non_outliers_df.loc[mask, ['MonetaryValue', 'Frequency', 'Recency', 'AOV', 'Tenure']])

- Lower tail outliers are genuine low-value customers — single purchases, minimal spend (£2.95–£167), largely inactive (recency 63–722 days). Real customers, not data errors — retained as they won't distort centroids meaningfully.

In [ ]:
plt.figure(figsize=(25, 5))

for i, col in enumerate(["MonetaryValue", "Frequency", "Recency", "AOV", "Tenure"], 1):
    plt.subplot(1, 5, i)
    sns.boxplot(data=transformed_df[col], color='skyblue')
    plt.title(f'{col} After Yeo-Johnson')

plt.tight_layout()
plt.show()

- Visualization confirms statistics only insignificant outliers have left for `AOV` and `MonetaryValue`

**Correlation Matrix - Multicolinearity**

In [ ]:
plt.figure(figsize=(10, 8))
corr = transformed_df[["MonetaryValue", "Frequency", "Recency", "AOV", "Tenure"]].corr()

sns.heatmap(
    corr,
    annot=True,
    fmt=".2f",
    cmap='coolwarm',
    vmin=-1,
    vmax=1,
    linewidths=0.5,
    square=True,
    cbar_kws={"shrink": .8}
)

plt.title("Correlation Matrix After Transformations")
plt.show()

- `Tenure` is highly correlated with `frequency` (0.88) and `monetary` (0.75).
- `Frequency` also has high multicolinearity with `monetary` (0.85) and `tenure` (0.75).

**Conclusions**
- From the data standpoint both `frequency` and `tenure` are problematic. `Tenure` has a massive peak and is heavily rightly skewed and is highly multicolinear with other features, whereas `frequency` is mid-higher rightly skewed, but has a gradual decrease, and has insignificantly smaller multicolinearity with other features, but still high. Out of this two from the data standpoint `frequency` is better, especially given gradual decrease.

- From the business standpoint, `tenure` is a passive metric. It tells how long someone has been a customer, but it doesn't indicate their current intent to buy or their value, so it is less valuable for marketing team to target, whereas `frequency` is a standard for rfm clustering, and most importantly after further experimentation in this project without it and important distinction of two clusters and following the strategy for them two would be merged.
- Conclusion: `tenure` should be dropped.
- Additionaly `frequency` multicolinearity can be handled for K-Means with PCA.

**Dropping Tenure**

In [ ]:
transformed_df.drop(columns=["Tenure"], inplace=True)

transformed_df

## PCA COMPONENTS ANALYSIS AND SELECTION

**Variance**

In [ ]:
# Fitting PCA on all components to determine how many explain sufficient variance before reducing.

pca_check = PCA()
pca_check.fit(transformed_df)
print(np.cumsum(pca_check.explained_variance_ratio_))

- 3 components explain almost 100% of the data.
- overall one dimention will be dropped, but that is not a main point, most importantly the multicolinearity between problematic features will decrease .

**Fitting**

In [ ]:
# Fitting PCA with 3 components and inspecting loadings

pca = PCA(n_components=3)
pca_data = pca.fit_transform(transformed_df)

loadings = pd.DataFrame(
    pca.components_,
    columns=transformed_df.columns,
    index=['PC1', 'PC2', 'PC3']
)
print(loadings)
print(f"\nExplained variance ratio: {pca.explained_variance_ratio_}")

- PC1 (60.6%) — overall value/activity axis. High monetary, frequency, AOV, low recency.

- PC2 (24.5%) — spend pattern axis. Dominated by AOV, contrasts frequent low-spend vs infrequent high-spend.

- PC3 (14.7%) — recency/frequency axis. Dominated by recency and frequency, mostly independent of spend magnitude.

**Shape**

In [ ]:
aspect_ratio = pca.explained_variance_[0] / pca.explained_variance_[-1]
print(f"Overall aspect ratio: {aspect_ratio:.2f}")

- Overall data aspect ratio: **4.12** — the data is inherently elongated and non-spherical. This measures global spread, not cluster-level shape — per-cluster aspect ratios must be examined after fitting to confirm. If clusters are also non-spherical, GMM could be a better alternative.

<h2 style="text-align: center">CLUSTERING ALGORITHM CANDIDATE</h2>

## K-Means

**Global Silhouette**

In [ ]:
max_k = 8
silhouette_scores = []
k_values = range(2, max_k + 1)

for k in k_values:
    kmeans = KMeans(n_clusters=k, random_state=42, max_iter=50, n_init=50)
    cluster_labels = kmeans.fit_predict(pca_data)
    silhouette_scores.append(silhouette_score(pca_data, cluster_labels))

plt.plot(k_values, silhouette_scores, marker='o', color='orange')
plt.title('Silhouette Scores for Different Values of k')
plt.xlabel('Number of Clusters (k)')
plt.ylabel('Silhouette Score')
plt.xticks(k_values)
plt.grid(True)
plt.show()

- K=2: 0.366 — best, but business-useless.
- K=3: 0.302 — sharp drop, but compared to the next ones relatively high.
- K=4: 0.287 — bad candidate, local drop
- K=5: 0.301 — local peak, best potential candidate.
- K=6: 0.293 — slight drop, but is higher than 4.
- K=7: 0.281 — weakest overall.
- K=8: 0.294 — slight recovery, unexplained.

**Davies-Bouldin**

In [ ]:
davies_bouldin_scores = []

for k in k_values:
    kmeans = KMeans(n_clusters=k, random_state=42, max_iter=50, n_init=50)
    cluster_labels = kmeans.fit_predict(pca_data)
    davies_bouldin_scores.append(davies_bouldin_score(pca_data, cluster_labels))

plt.plot(k_values, davies_bouldin_scores, marker='o', color='red')
plt.title('Davies-Bouldin Scores for Different Values of k')
plt.xlabel('Number of Clusters (k)')
plt.ylabel('Davies-Bouldin Score')
plt.xticks(k_values)
plt.grid(True)
plt.show()

- K=2: 1.081 — good score, but still business-useless.
- K=3: 1.197 — worst of all, sharp spike. (silhouette rewards k=3 because fewer, larger clusters appear more cohesive on average, while Davies-Bouldin penalizes it because those large clusters are actually overlapping and poorly separated internally.)
- K=4: 1.166 — slightly better than 3, but still poor.
- K=5: 1.057 — best practical candidate, clear drop.
- K=6: 1.097 — worse than k=5.
- K=7: 1.043 — better than k=5 statistically, but over-segmentation risk.
- K=8: 1.009 — best score overall, but clearly over-segmented.

**Per-cluster silhouette**

In [ ]:
for k in [4, 5, 6]:
    kmeans = KMeans(n_clusters=k, random_state=42, max_iter=50, n_init=50)
    cluster_labels = kmeans.fit_predict(pca_data)
    sample_sil_values = silhouette_samples(pca_data, cluster_labels)
    sns.boxplot(x=cluster_labels, y=sample_sil_values)
    plt.title(f"Per-Cluster Silhouette (k={k})")
    plt.show()

- K=4: All clusters above 0.0, no negatives. Clusters 0 and 1 solid (medians ~0.32, 0.39), clusters 2 and 3 weaker (~0.25). Most consistent of the three candidates.

- K=5: Cluster 2 strongest (median ~0.41), some have negative values

- K=6: No cluster collapses, all medians above 0.2. More consistent than k=5.


**Final Decision:**
After narrowing candidates to k=4–6, k=6 is eliminated as the weakest. k=4 shows cleaner per-cluster silhouette, but Davies-Bouldin and global silhouette both favor k=5. Given the data is elongated and non-spherical, Davies-Bouldin is the more reliable metric here — and it clearly points to k=5. Additionally, k=5 introduces an extra segment with potential business value, making it a better choice.

**Initialization Sensitivity Check** 

In [ ]:
base = KMeans(n_clusters=5, random_state=42, max_iter=50, n_init=50).fit_predict(pca_data)

for seed in [0, 7, 123, 999]:
    labels = KMeans(n_clusters=5, random_state=seed, max_iter=50, n_init=50).fit_predict(pca_data)
    print(f"seed={seed}: ARI={adjusted_rand_score(base, labels):.4f}")

- Cluster assignments are highly stable across all seeds (ari ≥ 0.99). N_init=50 ensures the algorithm consistently escapes local optima. The model is robust and deterministic in practice.

**Convergence check**

In [ ]:
for max_iter in [10, 20, 30, 40, 50]:
    km = KMeans(n_clusters=5, random_state=42, max_iter=max_iter, n_init=50)
    km.fit(pca_data)
    print(f"max_iter={max_iter}: converged in {km.n_iter_} iterations")

- Although converges at 14 and max_iter=20 would be enough, 50 is a nice safety net and costs almost nothing.

**Stability analysis**

In [ ]:
N = 100
ari_scores = []
reference = KMeans(n_clusters=5, random_state=42, max_iter=50, n_init=50).fit_predict(pca_data)

for i in range(N):
    sample = resample(pca_data, replace=True, random_state=i)
    labels = KMeans(n_clusters=5, random_state=42, max_iter=50, n_init=50).fit_predict(sample)
    
    # Compare only on the indices that were sampled
    idx = resample(np.arange(len(pca_data)), replace=True, random_state=i)
    ari_scores.append(adjusted_rand_score(reference[idx], labels))

print(f"Bootstrap Stability — Mean ARI: {np.mean(ari_scores):.4f} ± {np.std(ari_scores):.4f}")

- Bootstrap stability (100 iterations) have a mean ARI of 0.857 ± 0.082 — relatvely stable reproducibility.

## Final cluster assignment

In [ ]:
kmeans = KMeans(n_clusters=5, random_state=42, max_iter=50, n_init=50)
cluster_labels = kmeans.fit_predict(pca_data)
cluster_labels

In [ ]:
non_outliers_df["Cluster"] = cluster_labels
non_outliers_df

## CLUSTER-LEVEL SPHERICALITY CHECK

In [ ]:
for cluster_id in range(5):
    mask = non_outliers_df['Cluster'] == cluster_id
    pts = pca_data[mask]
    cov = np.cov(pts.T)
    vals = np.linalg.eigvalsh(cov)
    aspect_ratio = vals.max() / vals.min()
    print(f"Cluster {cluster_id}: aspect ratio = {aspect_ratio:.2f}")

- All individual clusters are confirmed to be non-spherical and are problematic for K-Means

In [ ]:
colors = plt.cm.tab10.colors

fig, ax = plt.subplots(figsize=(10, 7))

for cluster_id in range(5):
    mask = non_outliers_df['Cluster'] == cluster_id
    pts = pca_data[mask, :2]
    color = colors[cluster_id]
    ax.scatter(pts[:, 0], pts[:, 1], s=10, alpha=0.4, label=f'Cluster {cluster_id}', color=color)
    cov = np.cov(pts.T)
    vals, vecs = np.linalg.eigh(cov)
    vals, vecs = vals[::-1], vecs[:, ::-1]
    angle = np.degrees(np.arctan2(*vecs[:, 0][::-1]))
    w, h = 2 * 2 * np.sqrt(vals)
    ax.add_patch(Ellipse(xy=pts.mean(axis=0), width=w, height=h, angle=angle,
                         edgecolor=color, facecolor='none', linewidth=2))

ax.set_xlabel('PC1')
ax.set_ylabel('PC2')
ax.set_title('PCA Clusters with Covariance Ellipses')
ax.legend()
plt.show()

- The clusters are not spherical — the ellipses are clearly elongated/tilted, indicating non-spherical (elliptical) shapes with different orientations and sizes - visualization confirms - all of this indicates that GMM could potentially work better.

## GMM Model Test

**Parameters search**

In [ ]:
from sklearn.metrics import silhouette_score, davies_bouldin_score

results = []

for cov_type in ['spherical', 'tied', 'diag', 'full']:
    for n in range(2, 9):
        gmm = GaussianMixture(n_components=n, covariance_type=cov_type, random_state=42)
        gmm.fit(transformed_df)
        labels = gmm.predict(transformed_df)
        sil = silhouette_score(transformed_df, labels)
        db = davies_bouldin_score(transformed_df, labels)
        results.append({'n': n, 'cov': cov_type, 'silhouette': round(sil, 3), 'db': round(db, 3)})

results_df = pd.DataFrame(results)

print("Best Silhouette:")
print(results_df.loc[results_df['silhouette'].idxmax()])

print("\nBest Davies-Bouldin:")
print(results_df.loc[results_df['db'].idxmin()])

print("\nFull results:")
print(results_df.to_string(index=False))

- KMeans wins silhouette (0.301 vs 0.280), GMM wins Davies-Bouldin (1.000 vs 1.057). The difference is marginal — essentially a tie. Not enough to justify switching from KMeans.
- Spherical GMM won here reinforces that clusters behave more spherically than the aspect ratios suggested — likely because PowerTransformer + the natural RFM separation made them compact enough for spherical assumptions to hold reasonably well. It means  data doesn't actually need GMM's elliptical modeling — KMeans' spherical assumption is appropriate and even better from the bussiness perspective because KMeans is better when segments are clear-cut and you are needed actionable hard labels for marketing team - which is a purpose of this project.

**Note:** PCA was introduced for KMeans (which assumes feature independence), but applying it before GMM would destroy the elliptical correlation structure that GMM is specifically designed to exploit — that's why GMM is fit on transformed_df directly, without PCA.

## CLUSTER NAMING

In [ ]:
print(non_outliers_df.groupby("Cluster")[["MonetaryValue", "Frequency", "Recency", "AOV"]].mean())

**QUICK SUMMARY OF THE RESULTS ABOVE**:

- Cluster 0 — mid spend, mid frequency, very recent, low aov → `Promising` (frequent and recent but lower spend per order).

- Cluster 1 — high spend, high frequency, recent, high aov → `Vip` (best customers, active, frequent, high value).

- Cluster 2 — low spend, low frequency, very high recency, low aov → `Churned` (inactive, rare, low value).

- Cluster 3 — mid spend, mid frequency, high recency, low aov → `At-risk frequent` (was a regular buyer, now disengaged — needs volume incentives).

- Cluster 4 — mid spend, low frequency, high recency, high aov → `At-risk high-value` (rare but high-spend buyer, now disengaged — needs premium re-engagement).

In [ ]:
km_k4 = KMeans(n_clusters=4, random_state=42, max_iter=50, n_init=50)
labels_k4 = km_k4.fit_predict(pca_data)
raw_with_k4 = aggregated_df.loc[non_outliers_df.index].copy()
raw_with_k4["Cluster"] = labels_k4
print(raw_with_k4.groupby("Cluster")[["MonetaryValue", "Frequency", "Recency", "AOV"]].mean())

K=4 would lose At-Risk Frequent entirely — it would get absorbed into Promising (Cluster 3, recency 99 days) despite being a disengaged segment. At-Risk High-Value (Cluster 2, recency 332, AOV £390) survives but without its counterpart. The two segments that k=5 distinguishes:

- `AOV`: £426 vs £216 — at-risk high-value spends 2× per order.

- `Frequency`: 1.52 vs 4.33 — at-risk high-value was always a rare buyer, at-risk frequent was a regular buyer now disengaged.

Merging them leads to wrong marketing strategy — at-risk frequent needs volume incentives, at-risk high-value needs premium re-engagement. K=5 preserves this distinction.

## CLUSTER LABEL ASSIGNMENT

In [ ]:
cluster_means = non_outliers_df.groupby("Cluster")[["MonetaryValue", "Frequency", "Recency", "AOV"]].mean()

vip_cluster = cluster_means["MonetaryValue"].idxmax()
churned_cluster = cluster_means["Recency"].idxmax()
remaining = cluster_means.drop([vip_cluster, churned_cluster])
atrisk_hv_cluster = remaining["AOV"].idxmax()
remaining2 = remaining.drop([atrisk_hv_cluster])
atrisk_freq_cluster = remaining2["Recency"].idxmax()
promising_cluster = [c for c in cluster_means.index if c not in [vip_cluster, churned_cluster, atrisk_hv_cluster, atrisk_freq_cluster]][0]

cluster_labels_names = {
    vip_cluster: "VIP",
    churned_cluster: "Churned",
    atrisk_hv_cluster: "At-Risk High-Value",
    atrisk_freq_cluster: "At-Risk Frequent",
    promising_cluster: "Promising"
}

cluster_colors = {
    'VIP': '#ff7f0e',
    'Churned': '#1f77b4',
    'At-Risk High-Value': '#2ca02c',
    'At-Risk Frequent': '#9467bd',
    'Promising': '#d62728'
}

## Distributional Consistency Check

In [ ]:
SPLIT_DATE = "2010-12-01"

for name in ["H1", "H2"]:
    subset = cleaned_df[cleaned_df["InvoiceDate"] < SPLIT_DATE] if name == "H1" else cleaned_df[cleaned_df["InvoiceDate"] >= SPLIT_DATE]
    subset = subset.copy()
    subset["SalesLineTotal"] = subset["Quantity"] * subset["Price"]
    ref = subset["InvoiceDate"].max()
    agg = subset.groupby("Customer ID").agg(
        MonetaryValue=("SalesLineTotal", "sum"),
        Frequency=("Invoice", "nunique"),
        LastInvoiceDate=("InvoiceDate", "max")
    )
    agg["Recency"] = (ref - agg["LastInvoiceDate"]).dt.days
    agg["AOV"] = agg["MonetaryValue"] / agg["Frequency"]
    agg["Tenure"] = (agg["LastInvoiceDate"] - subset.groupby("Customer ID")["InvoiceDate"].min()).dt.days
    outlier_mask = (
        (agg["MonetaryValue"] > M_Q3   + 1.5 * M_IQR) |
        (agg["Frequency"]     > F_Q3   + 1.5 * F_IQR) |
        (agg["AOV"]           > AOV_Q3 + 1.5 * AOV_IQR)
    )
    agg_clean = agg[~outlier_mask]
    transformed = pt.transform(agg_clean[["MonetaryValue", "Frequency", "Recency", "Tenure", "AOV"]])
    pca_input = pca.transform(
        pd.DataFrame(transformed, columns=["MonetaryValue", "Frequency", "Recency", "Tenure", "AOV"])
        [["MonetaryValue", "Frequency", "Recency", "AOV"]]
    )
    labels = pd.Series(kmeans.predict(pca_input)).map(cluster_labels_names).value_counts(normalize=True).mul(100).round(1)
    print(f"\n{name}:\n{labels}")

    # TENURE IS COMPUTED SOLELY TO SATISFY THE POWER 
    # TRANSFORMER FITTED ON 5 FEATURES — IT IS DROPPED BEFORE PCA AND PLAYS NO ROLE IN CLUSTERING.

- Distributions are highly consistent across two similar-length windows (dec 2009–nov 2010 vs dec 2010–dec 2011). No segment shifts by more than 1.6 percentage points. Note: pt, pca, and kmeans were trained on the full dataset — this confirms the trained model's outputs are stable across time slices.

## CLUSTER VISUALIZATION

In [ ]:
colors = non_outliers_df['Cluster'].map(cluster_labels_names).map(cluster_colors)

fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(projection='3d')

ax.scatter(pca_data[:, 0], 
           pca_data[:, 1], 
           pca_data[:, 2],
           c=colors,
           marker='o',
           alpha=0.5, s=10)

ax.set_xlabel('PC1')
ax.set_ylabel('PC2')
ax.set_zlabel('PC3')
ax.set_title('3D Scatter Plot of Customer Clusters in PCA Space')

legend_elements = [Patch(facecolor=color, label=label) 
for label, color in cluster_colors.items()]

ax.legend(handles=legend_elements)

plt.tight_layout()
plt.show()

- Vip (orange) — clear separation on the right, well distinct.

- Churned (blue) — sits mostly on the left, reasonably distinct.

- Promising (red) — sits below vip, partially overlaps on pc1 but separated on pc2/pc3.

- At-risk high-value (green) and at-risk frequent (purple) — heavily overlap with each other and with churned in the center-left area.

- Note: overlap is expected for customer segmentation data — distinction is business-driven rather than purely algorithmic. Confirms weak but acceptable silhouette scores.

## OUTLIER CLUSTER ASSIGNMENT

In [ ]:
outlier_clusters_df = pd.concat([monetary_outliers_df, frequency_outliers_df, aov_outliers_df]).drop_duplicates()

print(outlier_clusters_df[["MonetaryValue", "Frequency", "Recency", "AOV"]].describe())

outlier_transformed = pt.transform(outlier_clusters_df[["MonetaryValue", "Frequency", "Recency", "Tenure", "AOV"]])
outlier_transformed_df = pd.DataFrame(outlier_transformed, columns=["MonetaryValue", "Frequency", "Recency", "Tenure", "AOV"])
outlier_transformed_df = outlier_transformed_df[["MonetaryValue", "Frequency", "Recency", "AOV"]]

outlier_pca = pca.transform(outlier_transformed_df)
outlier_clusters_df["Cluster"] = kmeans.predict(outlier_pca)

## FULL DATASET CLUSTER ASSIGNMENT

In [ ]:
full_clustering_df = pd.concat([
    non_outliers_df[["Cluster"]],
    outlier_clusters_df[["Cluster"]]
])

full_clustering_df["ClusterLabel"] = full_clustering_df["Cluster"].map(cluster_labels_names)
full_clustering_df

## WHALE IDENTIFICATION

In [ ]:
full_clustering_df["IsWhale"] = (
    full_clustering_df.index.isin(outlier_clusters_df[outlier_clusters_df["Cluster"] == vip_cluster].index)
)

- Vip sub-segment: whales (| statistical outliers | b2b/wholesale buyers).

In [ ]:
# VIP vs Whale comparison

print(f"Non-outliers VIP: {(non_outliers_df['Cluster'] == vip_cluster).sum()}")
print(f"Outliers assigned to VIP: {(outlier_clusters_df['Cluster'] == vip_cluster).sum()}")
print(non_outliers_df[non_outliers_df["Cluster"] == vip_cluster][["MonetaryValue", "Frequency", "Recency", "AOV"]].describe())
print(outlier_clusters_df[outlier_clusters_df["Cluster"] == vip_cluster][["MonetaryValue", "Frequency", "Recency", "AOV"]].mean())

## CLUSTER FEATURE DISTRIBUTION

In [ ]:
violin_df = aggregated_df.loc[aggregated_df.index.isin(full_clustering_df.index)].join(full_clustering_df[['ClusterLabel', 'IsWhale']], how='inner')

violin_df["ClusterLabel_viz"] = violin_df.apply(lambda r: "VIP (Whale)" if r["IsWhale"] else r["ClusterLabel"], axis=1)
cluster_colors_viz = {**cluster_colors, "VIP (Whale)": "#ff0000"}

plt.figure(figsize=(12, 16))
plt.subplot(4, 1, 1)
sns.violinplot(x=violin_df['ClusterLabel_viz'], y=violin_df['MonetaryValue'], hue=violin_df['ClusterLabel_viz'], palette=cluster_colors_viz, legend=False)
plt.title('Monetary Value by Cluster')
plt.ylabel('Monetary Value (£)')
plt.ylim(0, violin_df['MonetaryValue'].quantile(0.95))
plt.subplot(4, 1, 2)
sns.violinplot(x=violin_df['ClusterLabel_viz'], y=violin_df['Frequency'], hue=violin_df['ClusterLabel_viz'], palette=cluster_colors_viz, legend=False)
plt.title('Frequency by Cluster')
plt.ylabel('Frequency (# orders)')
plt.ylim(0, violin_df['Frequency'].quantile(0.95))
plt.subplot(4, 1, 3)
sns.violinplot(x=violin_df['ClusterLabel_viz'], y=violin_df['Recency'], hue=violin_df['ClusterLabel_viz'], palette=cluster_colors_viz, legend=False)
plt.title('Recency by Cluster')
plt.ylabel('Recency (days)')
plt.subplot(4, 1, 4)
sns.violinplot(x=violin_df['ClusterLabel_viz'], y=violin_df['AOV'], hue=violin_df['ClusterLabel_viz'], palette=cluster_colors_viz, legend=False)
plt.title('AOV by Cluster')
plt.ylabel('AOV (£)')
plt.ylim(0, violin_df['AOV'].quantile(0.95))
plt.tight_layout()
plt.show()

- `Monetary` — vip clearly highest, churned lowest, at-risk high-value surprisingly high due to outliers. Well separated.

- `Frequency` — vip highest, at-risk high-value and churned very low. At-risk frequent shows wider distribution — confirms mid-`frequency` buyers. Vip (whale) shows extreme `frequency` — confirms b2b/wholesale behavior.

- Recency — churned and at-risk both high (disengaged), vip and vip (whale) lowest (most recent), promising very low. At-risk high-value and at-risk frequent overlap heavily on recency — key distinction is `aov` and `frequency`.

- `Aov` — vip (whale) clearly highest, confirming extreme per-order spend. At-risk high-value and core vip both high, at-risk frequent, promising and churned lower. `Aov` is what separates at-risk high-value from at-risk frequent — key distinguishing feature.

## STATISTICAL SEPARATION VALIDATION

In [ ]:
n = len(violin_df)
for col in ["MonetaryValue", "Frequency", "Recency", "AOV"]:
    groups = [violin_df[violin_df["ClusterLabel_viz"] == label][col] for label in violin_df["ClusterLabel_viz"].unique()]
    stat, p = kruskal(*groups)
    epsilon_squared = stat / (n - 1)
    print(f"{col}: H={stat:.2f}, p={p:.2e}, ε²={epsilon_squared:.3f}")

- All four features differ significantly across segments (p ≈ 0 for all). Kruskal-Wallis confirms cluster separation is statistically real.
- Effect sizes are large across all features (ε² = 0.61–0.82, threshold for large effect is 0.14) — differences are practically meaningful, not a sample size artifact.

## CLUSTER PROFILE OVERVIEW

In [ ]:
cluster_counts = violin_df['ClusterLabel_viz'].value_counts()

feature_means = violin_df.groupby('ClusterLabel_viz')[['Recency', 'Frequency', 'MonetaryValue', 'AOV']].mean()
feature_means = feature_means.reindex(cluster_counts.index)

feature_means_viz = (feature_means - feature_means.min()) / (feature_means.max() - feature_means.min())
feature_means_viz = feature_means_viz.reindex(cluster_counts.index)

fig, ax1 = plt.subplots(figsize=(12, 8))
sns.barplot(x=cluster_counts.index, y=cluster_counts.values, ax=ax1, palette='viridis', hue=cluster_counts.index)
ax1.set_ylabel('Number of Customers', color='b')
ax1.set_title('Cluster Distribution with Average Feature Values')

ax2 = ax1.twinx()
sns.lineplot(data=feature_means_viz, ax=ax2, palette='Set2', marker='o')
ax2.set_ylabel('Normalized Average Value (0-1)', color='g')

plt.show()

- Vip (1,338 customers) — largest core segment, highest `frequency`, lowest recency, high `monetary`.

- Vip (whale) (722 customers) — highest `monetary` and `aov` — extreme spenders. Lowest recency — most active. Confirms b2b/wholesale behavior.

- At-risk high-value (1,074) — high `aov`, very high recency — disengaged high-spenders.

- Churned (1024) — highest recency, lowest `frequency` and `monetary` — inactive low-value.

- Promising (855) — low recency (active), low `monetary` and `frequency` — regular but low-spend.

- At-risk frequent (839) — high recency, mid `aov` (£216), highest `frequency` among at-risk — formerly regular buyers now disengaged.

In [ ]:
print(feature_means)

In [ ]:
revenue_share = violin_df.groupby('ClusterLabel_viz')['MonetaryValue'].sum()
revenue_share_pct = (revenue_share / revenue_share.sum() * 100).round(1)
print(revenue_share_pct)

- Revenue share: vip (whale) 64.8%, vip (core) 19.3%, at-risk high-value 7.9%, at-risk frequent 4.5%, promising 2.6%, churned 1.0%. Whales alone generate 64.8% of total revenue — dedicated b2b account management is the single highest-priority action. Core vip retention is second. At-risk high-value re-engagement third.

In [ ]:
print(violin_df['ClusterLabel_viz'].value_counts())

**FINAL BUSINESS RECOMMENDATIONS**:

- Vip core (1,338 customers | £2,461 avg spend | 7.5 orders | 55 days recency | £355 `aov`) — retain and reward. Loyalty programs, early access, exclusive previews.

- Vip whale (722 customers | £15,312 avg spend | 24.4 orders | 43 days recency | £662 `aov`) — dedicated account managers, b2b/wholesale pricing, volume discount contracts, net-30/60 payment terms, direct support channel. Do not treat with mass campaigns.

- Promising (855 customers | £521 avg spend | 2.71 orders | 31 days recency) — nurture and upsell. Most recently active segment. Cross-sell complementary products and introduce premium lines to migrate toward vip.

- At-risk high-value (1,074 customers | £1,252 avg spend | 1.52 orders | 338 days recency) — re-engage urgently. Highest `aov` (£740) — they spend big when they buy. Personalized win-back campaigns and exclusive discounts before full churn.

- At-risk frequent (839 customers | £909 avg spend | 4.40 orders | 299 days recency) — volume incentives. Were regular buyers — bundle deals and "we miss you" campaigns to reactivate purchase habits.

- Churned (1,024 customers | £165 avg spend | 1.25 orders | 410 days recency) — low-cost re-engagement only. Automated email with strong incentives. Deprioritize if no response after 2-3 attempts.

<h2 style="text-align: center">ESTIMATED BUSINESS IMPACT</h2>

- Illustrative win-back revenue potential per segment: customers × industry-benchmark 
reactivation rate × AOV (value of one recovered order). Rates are illustrative 
industry benchmarks, not fitted to this data.

In [ ]:
recovery_scenarios = {
    "At-Risk High-Value": {"customers": 1074, "aov": 740, "rates": [0.05, 0.10, 0.15]},
    "At-Risk Frequent":   {"customers": 839,  "aov": 207, "rates": [0.05, 0.10, 0.15]},
    "Churned":            {"customers": 1024, "aov": 127, "rates": [0.02, 0.035, 0.05]},
}

for segment, d in recovery_scenarios.items():
    low, mid, high = [d["customers"] * r * d["aov"] for r in d["rates"]]
    print(f"{segment}: low=£{low:,.0f} mid=£{mid:,.0f} high=£{high:,.0f}")

- Mid-case total: ~£101k in potential recovered revenue across at-risk and churned segments (£79,476 + £17,367 + £4,552). Illustrative estimate based on industry-benchmark reactivation rates — not validated against actual campaign performance on this dataset.

<h2 style="text-align: center">INFERENCE DEMO</h2>

In [ ]:
joblib.dump(pt, '../artifacts/power_transformer.pkl')
joblib.dump(pca, '../artifacts/pca.pkl')
joblib.dump(kmeans, '../artifacts/kmeans_k5.pkl')
joblib.dump(max_invoice_date, '../artifacts/reference_date.pkl')
joblib.dump(cluster_labels_names, '../artifacts/cluster_labels_names.pkl')
joblib.dump({
    'M_Q1': M_Q1, 'M_Q3': M_Q3, 'M_IQR': M_IQR,
    'F_Q1': F_Q1, 'F_Q3': F_Q3, 'F_IQR': F_IQR,
    'AOV_Q1': AOV_Q1, 'AOV_Q3': AOV_Q3, 'AOV_IQR': AOV_IQR
}, '../artifacts/iqr_bounds.pkl')              

In [ ]:
# Load artifacts
pt = joblib.load('../artifacts/power_transformer.pkl')
pca = joblib.load('../artifacts/pca.pkl')
kmeans = joblib.load('../artifacts/kmeans_k5.pkl')
iqr_bounds = joblib.load('../artifacts/iqr_bounds.pkl')
reference_date = joblib.load('../artifacts/reference_date.pkl')
cluster_labels_names = joblib.load('../artifacts/cluster_labels_names.pkl')

# New customers (raw values)
customers_raw = pd.DataFrame([
    {'last_purchase': '2011-11-20', 'first_purchase': '2011-01-01', 'MonetaryValue': 3000.0, 'Frequency': 10, 'AOV': 400.0},   # VIP
    {'last_purchase': '2011-11-08', 'first_purchase': '2011-06-01', 'MonetaryValue': 521.0,  'Frequency': 3,  'AOV': 210.0},   # Promising
    {'last_purchase': '2011-07-01', 'first_purchase': '2011-04-01', 'MonetaryValue': 700.0,  'Frequency': 1,  'AOV': 600.0},   # At-Risk High-Value
    {'last_purchase': '2011-06-01', 'first_purchase': '2011-05-01', 'MonetaryValue': 150.0,  'Frequency': 1,  'AOV': 80.0},    # Churned
    {'last_purchase': '2011-04-01', 'first_purchase': '2010-12-01', 'MonetaryValue': 900.0,  'Frequency': 4,  'AOV': 225.0},   # At-Risk Frequent
    {'last_purchase': '2011-09-01', 'first_purchase': '2011-05-01', 'MonetaryValue': 600.0,  'Frequency': 2,  'AOV': 450.0},   # Borderline
    {'last_purchase': '2011-11-20', 'first_purchase': '2010-12-01', 'MonetaryValue': 10000.0,'Frequency': 100,'AOV': 1000.0},  # Outlier
])

customers_raw['last_purchase'] = pd.to_datetime(customers_raw['last_purchase'])
customers_raw['first_purchase'] = pd.to_datetime(customers_raw['first_purchase'])
customers_raw['Recency'] = (reference_date - customers_raw['last_purchase']).dt.days
customers_raw['Tenure'] = (customers_raw['last_purchase'] - customers_raw['first_purchase']).dt.days
customers = customers_raw[['MonetaryValue', 'Frequency', 'Recency', 'Tenure', 'AOV']]


# Outlier flag
def is_outlier(row):
    return (
        row['MonetaryValue'] > iqr_bounds['M_Q3'] + 1.5 * iqr_bounds['M_IQR'] or
        row['Frequency']     > iqr_bounds['F_Q3'] + 1.5 * iqr_bounds['F_IQR'] or
        row['AOV']           > iqr_bounds['AOV_Q3'] + 1.5 * iqr_bounds['AOV_IQR']
    )

customers_raw['is_outlier'] = customers.apply(is_outlier, axis=1)

# Transform
transformed = pt.transform(customers)
transformed_df = pd.DataFrame(transformed, columns=['MonetaryValue', 'Frequency', 'Recency', 'Tenure', 'AOV'])
transformed_df = transformed_df[['MonetaryValue', 'Frequency', 'Recency', 'AOV']]

# PCA + predict
pca_input = pca.transform(transformed_df)
clusters = kmeans.predict(pca_input)

labels = ['VIP', 'Promising', 'At-Risk High-Value', 'Churned', 'At-Risk Frequent', 'Borderline','Outlier']
for i, cluster in enumerate(clusters):
    outlier_flag = '⚠️ OUTLIER' if customers_raw['is_outlier'].iloc[i] else ''
    whale_flag = '🐋 WHALE' if (customers_raw['is_outlier'].iloc[i] and cluster_labels_names[cluster] == 'VIP') else ''
    print(f"Customer {i+1} ({labels[i]}): Cluster {cluster} — {cluster_labels_names[cluster]} {outlier_flag} {whale_flag}")

- All 7 targeted customers landed in their intended clusters, confirming the inference pipeline works correctly end-to-end.

- Customer 7 (outlier) correctly classified as vip and flagged with ⚠️ outlier — outlier detection pipeline works.

- Borderline case (customer 6): high `aov` (£450) and low `frequency` (2) assigned to at-risk high-value despite moderate recency — aligns with the cluster's core trait.

<h2 style="text-align: center">EXPORT</h2>

In [ ]:
# Clustered customer data for marketing team
violin_df.to_excel('../outputs/customer_segments.xlsx', index=True)

# Cluster summary statistics
feature_means.to_excel('../outputs/cluster_summary.xlsx', index=True)

# Normalized cluster summary (0-100 scale, for comparing all features on one chart)
feature_means_normalized = (feature_means - feature_means.min()) / (feature_means.max() - feature_means.min()) * 100
feature_means_normalized = feature_means_normalized.reset_index()
feature_means_normalized.columns = ['ClusterLabel_viz', 'Recency', 'Frequency', 'MonetaryValue', 'AOV']
feature_means_normalized.to_excel('../outputs/cluster_summary_normalized.xlsx', index=False)

- Export to excel — for marketing and management team.

<h2 style="text-align: center">LIMITATIONS</h2>

**LIMITATIONS:**

1. Data is from 2009-2011 — model may not generalize to current customer behavior (for that needs retraining).

2. `Monetary` = `AOV` × `Frequency` by construction, collapsing to r = 0.63 after Yeo-Johnson. Both are retained intentionally: `Monetary` separates VIP from all other segments, `AOV` distinguishes At-Risk High-Value from At-Risk Frequent. Removing either would collapse a meaningful business distinction and dedicated marketing strategies for their segments.

3. `max_invoice_date` is fixed at training time — all Recency values inflate as real time passes, causing customers to drift toward higher-recency segments without any change in actual behavior. Requires periodic retraining or a dynamic reference date in production.

Above limitations are explained more thoroughly in markdown cells in this project above.